## **Lab 3: Multi-Source Retail Sales Data Integration and Analysis**

### Domain: Retail Analytics

####Objective:
To import, clean, integrate and analyze retail sales data stored across
CSV, JSON and Excel files using R, and store the final integrated dataset
in a SQLite database for further analysis.

[GITHUB](https://github.com/PRASHIRAWAL/23102C0067_ProgramR_Problem-Statements/tree/main/Problem%20Statement%203)

In [8]:
# Installing and Loading Packages

install.packages(c(
  "tidyverse",
  "readxl",
  "jsonlite",
  "RSQLite",
  "DBI",
  "writexl"
))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [9]:
library(tidyverse)
library(readxl)
library(jsonlite)
library(RSQLite)
library(DBI)
library(writexl)

In [3]:
# Reading the original Excel dataset

retail <- read_excel("Online Retail.xlsx")

head(retail)
dim(retail)
str(retail)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


[1] 541909      8

tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [4]:
# Creating transaction.csv

transactions <- retail %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

write_csv(transactions, "transactions.csv")

head(transactions)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [5]:
# Creating products.json

products <- retail %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

write_json(
  products,
  "products.json",
  pretty = TRUE,
  auto_unbox = TRUE
)

head(products)

StockCode,Description,UnitPrice
<chr>,<chr>,<dbl>
85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
71053,WHITE METAL LANTERN,3.39
84406B,CREAM CUPID HEARTS COAT HANGER,2.75
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [10]:
# Creating customers.xlsx

customers <- retail %>%
  select(
    CustomerID,
    Country
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

write_xlsx(
  customers,
  "customers.xlsx"
)

head(customers)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


## **Task 1**

Import the CSV, JSON, and Excel datasets into R using suitable packages.

Inspect the datasets and perform necessary cleaning for:

* Missing values
* Duplicate records
* Invalid or zero quantities
* Invalid or zero unit prices



Create a new attribute:

**Revenue = Quantity x UnitPrice**

Briefly mention the cleaning decisions taken.

In [11]:
# Importing the CSV

transactions <- read_csv("transactions.csv")

head(transactions)
dim(transactions)

Rows: 541909 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): InvoiceNo, StockCode
dbl  (2): CustomerID, Quantity
dttm (1): InvoiceDate

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


[1] 541909      5

In [12]:
# Importing the JSON

products <- fromJSON("products.json")

head(products)
dim(products)

,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


[1] 4070    3

In [13]:
# Importing the EXCEL

customers <- read_excel("customers.xlsx")

head(customers)
dim(customers)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


[1] 4373    2

In [14]:
# Inspecting for missing values

colSums(is.na(transactions))

colSums(is.na(products))

colSums(is.na(customers))

sum(is.na(transactions))
sum(is.na(products))
sum(is.na(customers))

InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0

StockCode Description   UnitPrice 
          0         176           0

CustomerID    Country 
         1          0

[1] 135080

[1] 176

[1] 1

In [15]:
# Checking Duplicate Records

sum(duplicated(transactions))

sum(duplicated(products))

sum(duplicated(customers))

[1] 5429

[1] 0

[1] 0

In [16]:
# Removing them

transactions <- transactions %>%
  distinct()

products <- products %>%
  distinct()

customers <- customers %>%
  distinct()

In [17]:
# Cleaning invalid quantities

summary(transactions$Quantity)

      Min.    1st Qu.     Median       Mean    3rd Qu.       Max. 
-80995.000      1.000      3.000      9.623     10.000  80995.000 

In [18]:
sum(transactions$Quantity <= 0, na.rm = TRUE)

[1] 10513

In [19]:
transactions <- transactions %>%
  filter(
    !is.na(Quantity),
    Quantity > 0
  )

Negative quantities in this dataset can represent cancellations/returns. For a simple sales revenue analysis, we retain only positive quantities.

In [20]:
# Cleaning invalid prices

summary(products$UnitPrice)

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
    0.000     1.250     2.510     6.905     4.250 11062.060 

In [21]:
sum(products$UnitPrice <= 0, na.rm = TRUE)

[1] 215

In [22]:
products <- products %>%
  filter(
    !is.na(UnitPrice),
    UnitPrice > 0
  )

In [23]:
# Cleaning CustomerID

transactions <- transactions %>%
  filter(!is.na(CustomerID))

customers <- customers %>%
  filter(!is.na(CustomerID))

In [24]:
# Creating Revenue

transactions_clean <- transactions %>%
  left_join(
    products,
    by = "StockCode"
  )

In [25]:
transactions_clean <- transactions_clean %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

In [26]:
head(transactions_clean)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30


## **Task 2**

Integrate the Multiple Data Sources

Combine the transaction, product, and customer datasets using appropriate dplyrjoin operations such as left_join() or inner_join().

After integration:
* Verify the dimensions of the final dataset.

* Identify unmatched records, if any.

* Justify the type of join selected.

In [27]:
final_data <- transactions_clean %>%
  left_join(
    customers,
    by = "CustomerID"
  )

In [28]:
dim(final_data)

[1] 392708      9

In [29]:
head(final_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30,United Kingdom
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34,United Kingdom
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00,United Kingdom
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34,United Kingdom
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34,United Kingdom
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30,United Kingdom


In [30]:
# Checking unmatched records

sum(is.na(final_data$Country))

[1] 0

In [31]:
sum(is.na(final_data$Description))

[1] 4831

In [32]:
final_data %>%
  filter(is.na(Country)) %>%
  head()

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>


A left join was selected because all valid transaction records need to be retained while product and customer information is added wherever a matching StockCode or CustomerID exists. This also allows unmatched records to be identified instead of silently removing transactions.

## **Task 3**

Perform Sales and Customer Analysis

Using appropriate dplyr operations, determine:

1. Total sales revenue.
2. Top 5 products based on revenue.
3. Top 5 countries based on revenue
4. Top 5 customers based on total purchase value.


Using case_when(), classify customers into:

* Low Value
* Medium Value
* High Value
* Premium

Use suitable thresholds based on customer purchase values.

Finally, identify one high-performing market and one underperforming market andbriefly justify your observations.

In [33]:
# Total Sales Revenue
total_revenue <- final_data %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )


total_revenue

Total_Revenue
<dbl>
10752840


In [34]:
# Top 5 products
top_products <- final_data %>%
  group_by(StockCode, Description) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)


top_products

StockCode,Description,Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


In [36]:
# Top 5 countries
top_countries <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)


top_countries

Country,Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [37]:
# Top 5 customers
top_customers <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Purchase)) %>%
  slice_head(n = 5)


top_customers

CustomerID,Total_Purchase
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [38]:
# Customer value classification

## First calculate customer purchase values:

customer_value <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  )

In [39]:
q1 <- quantile(customer_value$Total_Purchase, 0.25)
q2 <- quantile(customer_value$Total_Purchase, 0.50)
q3 <- quantile(customer_value$Total_Purchase, 0.75)

In [40]:
customer_value <- customer_value %>%
  mutate(
    Value_Category = case_when(
      Total_Purchase <= q1 ~ "Low Value",
      Total_Purchase <= q2 ~ "Medium Value",
      Total_Purchase <= q3 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

In [41]:
customer_value %>%
  count(Value_Category)

Value_Category,n
<chr>,<int>
High Value,1084
Low Value,1085
Medium Value,1085
Premium,1085


In [42]:
head(customer_value)

CustomerID,Total_Purchase,Value_Category
<dbl>,<dbl>,<chr>
12346,77183.60,Premium
12347,5438.44,Premium
12348,1790.16,High Value
12349,1926.96,High Value
12350,408.04,Medium Value
12352,1616.57,High Value


In [43]:
# High-performing market
market_performance <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue))

In [44]:
head(market_performance, 1)

Country,Revenue
<chr>,<dbl>
United Kingdom,8861857


In [45]:
# Underperforming market
market_performance %>%
  filter(Revenue > 0) %>%
  arrange(Revenue) %>%
  slice_head(n = 1)

Country,Revenue
<chr>,<dbl>
Saudi Arabia,181.44


## **Task 4**

Store and Retrieve Data Using SQL

Create a SQLite database and export the final cleaned and integrated dataset to atable named retail_sales.

Execute any two meaningful SQL queries from R, such as:

* Top 5 customers based on revenue.
* Total revenue by country.

Conclude the analysis with three important business insights obtained from theresults.

In [46]:
con <- dbConnect(
  SQLite(),
  "retail_sales.db"
)

In [47]:
dbWriteTable(
  con,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

In [48]:
dbListTables(con)

[1] "retail_sales"

In [50]:
# SQL Query 1 — Top 5 Customers

query1 <- "
SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5;
"

top_5_customers_sql <- dbGetQuery(
  con,
  query1
)

top_5_customers_sql

CustomerID,Total_Revenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [51]:
# SQL Query 2 — Revenue by Country
query2 <- "
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC;
"


revenue_by_country_sql <- dbGetQuery(
  con,
  query2
)


head(revenue_by_country_sql, 10)

,Country,Total_Revenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


In [52]:
dbListFields(
  con,
  "retail_sales"
)

[1] "InvoiceNo"   "StockCode"   "CustomerID"  "Quantity"    "InvoiceDate"
[6] "Description" "UnitPrice"   "Revenue"     "Country"

In [53]:
dbGetQuery(
  con,
  "SELECT COUNT(*) AS Total_Rows FROM retail_sales"
)

Total_Rows
<int>
392708


In [54]:
dbDisconnect(con)

## Business Insights

1. The highest-revenue country contributes a significant portion of total
   sales, indicating that the company's revenue is concentrated in a few
   major markets.

2. The top-performing products generate substantially higher revenue than
   other products, indicating that these products are important contributors
   to overall business performance.

3. Customer purchase values vary considerably. The Premium and High Value
   customer groups represent important customers who can be targeted with
   loyalty programs and personalized offers.

## **Conclusion**

The experiment successfully demonstrated the integration of heterogeneous
retail data sources using R. Transaction data was imported from CSV, product
information from JSON, and customer information from Excel. The datasets were
cleaned by handling missing values, duplicate records, invalid quantities and
invalid prices.

The cleaned datasets were integrated using dplyr join operations and Revenue
was calculated using Quantity × UnitPrice. Sales, product, country and
customer-level analysis was then performed using group_by(), summarise(),
arrange(), and case_when().

Finally, the integrated dataset was stored in a SQLite database named
retail_sales.db and SQL queries were executed from R to retrieve meaningful
business information.

The analysis demonstrates how multi-source retail data can be transformed
into useful business insights using R and SQL.